# 3D-spectrospherics : why it's not straightforward

The problem is that... magnitude-phase decomposition is not *straightforwardly* applicaable in 3 dimensions. Let's see why in this notebook.

## Spherical harmonics

3d decomposition relies on a projection over a family of spherical harmonics. Using $\phi$ as the angle with x axis, and $\theta$ as the angle with the z axis, the spherical harmonic of degree $l$ and order $m$ is defined by 

$$
Y^m_l(\theta, \phi) = \begin{cases}
 P^{|m|}_l(\cos \theta)  \sin {(|m| \phi)} \ & m \lt 0 \\
 P^m_l(\cos \theta)  \cos {(m \phi)} \ &  m \geq 0
\end{cases}
$$

where $P^m_l(x)$ is the associated Legendre polynome of order $l$ and degree $m$ (corresponding to ambisonics' order, tricky disambiguation here). 



In [1]:
# mixing spherical harmonics in 3 dimensions
from spectrospherics import plot_spherical_harmonic

# plot_spherical_harmonic().servable()
plot_spherical_harmonic().show() # <- in browser version

Launching server at http://localhost:51194


## Rotation-invariance : rotation impacts coefficients


a 3D sound field can be then generated with a linear combination of these harmonics : 
$$
g(\theta, \phi) = \sum_{l=0}^{\infty} \sum_{m=-l}^{l} a_l^m Y^m_l(\theta, \phi)
$$

Hence, we can generate 3d sound fields by playing directly with the $a^m_l$ coefficients. 

In [1]:
# mixing spherical harmonics in 3 dimensions
from spectrospherics import plot_spherical_harmonics

plot_spherical_harmonics().servable()
# plot_spherical_harmonics().show() # <- in browser version

BokehModel(combine_events=True, render_bundle={'docs_json': {'dde38047-855f-4400-866a-4a6443df3c67': {'version…

However, you can notice that 

- each degree has $2m+1$ coefficients for each degree, not only 2. Hence, **a single polar decomposition by degree is not possible anymore**. 
- While a rotation of a degree's component in 2d was presevering $|c_n|$, that consists in a single real, in 3 dimensions it preserves the norm $\Vert \mathbf{a}_l \Vert = \sum_{l={-m}}^{m} (a_l^m)^2$. Hence, having a single number for the modulus does not make sense anymore. 


## Non-commutativity : an operational problem

Furthermore, let's have an operational approach for sound field deformation, and see what other problems it brings. 

**Euler angles.** The strength of the amplitude-phase spectrangular decomposition was that not only amplitude was rotation-invariant, but that *shifting the phase was direclty rotating the correponding circular harmonic*. Indeed, there is only one angle : a phase offset correspond to a single rotation. Clear and intuitive. 

In 3 dimensions, *it's not that simple*. Indeed, a rotation in 3 dimensions (an element of the $SO(3)$ group) can be generated by three successive rotations about coordinate axes. 
$$
\begin{align}
R(\alpha, \beta, \gamma) & = R_z(\alpha) R_y(\beta) R_z(\gamma) & \alpha, \gamma \in [0, 2 \pi ), \beta \in [0, \pi]
\end{align}
$$

where each angle has a clear geometric meaning : the pair $(\alpha, \beta)$ points the rotated pole sending each point $z$ to the direction with polar angle $\beta$ and azimuth $\alpha$, and the angle $\gamma$ being the **roll** about this direction, the extra degree of freedom that a direction does not have. For this reason, we need an Euler convention (ZYZ in the formulmation before) to identify a given rotation. 

**Non-commutativity.** $SO(3)$ is non-commutative, implying that $AB \neq BA$. Hence, **the ordering of rotations $(\alpha, \beta, \gamma)$ is essential**: 

$$
R_z(\alpha) R_y(\beta) R_z(\gamma) \neq R_z(\alpha)  R_z(\gamma) R_y(\beta)
$$


In [1]:
# a demonstration of Euler non-commutativity

from spectrospherics import plot_euler_noncommutativity

plot_euler_noncommutativity().servable()
# plot_euler_noncommutativity().show()  # <- in browser version

Launching server at http://localhost:60586


**A Lie group explanation.** Lie algebra is a common algebra used to deal with non-combinatorial mathematical groups. An important notion is the Lie bracket, that measures the infinitesimal error between different pairwise orders : 

$$
[A, B] = AB - BA
$$
such that $[A, B] = - [B, A]$. With $SO(3)$, we can demostrate that
$$
\begin{align}
 [J_x, J_y] &= i J_z &  [J_y, J_z] & = J_x & [J_z, J_x] & = J_y
\end{align}
$$
where $J_x, J_y, J_z$ describe infinitesimal rotations under its indicial axis. We can see that **doing a small $x$-turn then $y$, then $-y$ then $-x$, leaves a residual $z$-turn**.

Equivalently, the loop $R_x(\epsilon)R_y(\delta)R_x(-\epsilon)R_y(-\delta) \approx R_z(\epsilon\delta)$ : a tiny roun trip does not close a leave a small rotation about the third axis.

In [ ]:
# explaining Lie groups

## Combinatorics: higher degree means higher redundency. 

Finally, the last problem. In 3 dimensions, a higher degree comes with a higher number of harmonics. However... the number of dimensions is the same. We can feel that this comes with a linear algebra problem: provided the orthogonality of spherical harmonics of degree $m$, projecting $2m+1$ coordinates onto 3 must have some redundency for $m > 1$. 


Indeed, each harmonic can be transformed in another of same order by : 

$$
Y^m_l(\theta, \phi) = \sum_{m'=-l}^l [D^{(l)}_{mm'}(\mathcal{R})]^* Y^{m'}_l(\theta, \phi)
$$

where $$D^{(l)}_{mm'}(\mathcal{R})$$ is the Wigner D-matrix corresponding to the rotation defined on Euler angles $$\mathcal{R}(\alpha, \beta, \gamma)$$. Hence, a mode-wise rotation cannot be defined anymore, as it is now the norm $\vert \mathbf{a}_l$ that is preserved under rotation. Furthermore, because of the non-commutativess of 3-d rotation groupe $SO(3)$, rotations are unique under Euler angles, and not $(\phi, \theta)$ coordinates. And, finally, it will break the most integrated comfort zone of syntesizers : non-commutativity may provide two different sound fields with same angles, depending on the update order.


We will explore below several methods aiming to recover a similar algebraic or functional structure between the $2l+1$ coefficients for each order $l$. Before starting, we can first derive some general properties of this problem : 

First, we can obtain the number of independant rotations invariants per order, that is $(2l + 1)$ minus the dimension of orbit (): 
- if $l=1$, the orbit is 2, the dimension is 3, so there is one invariant - the *radius*.
- if $l \geq 2$, the orbit is 3, so there are $2l - 2$ invariants : the radius, plus $2l-3$ more.


So... how to face all of that? First, we will enter the real core of 3d rotations : the **Wigner d-matrix**. 